# Game 1 chronological validation

The later test cohort estimates performance after fitting on earlier players. Training and validation labels must be observable before the next cohort starts. One-event and multi-event groups use fixed count boundaries. The report uses the metrics printed by this notebook. Run 00_setup and Game 1 extraction first.

In [1]:
import os
import sys
from pathlib import Path

candidates = (
    [Path(os.environ["CHURN_PROJECT_ROOT"])] if "CHURN_PROJECT_ROOT" in os.environ else
    [Path.cwd() / ".pipeline", Path.cwd(), Path.cwd().parent / ".pipeline", Path.cwd().parent]
)
ROOT = next((candidate.resolve() for candidate in candidates
             if (candidate / "src" / "config.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run 00_setup.ipynb first or point CHURN_PROJECT_ROOT at the pipeline checkout.")
sys.path.insert(0, str(ROOT))

import importlib.metadata
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score
from sklearn.metrics import roc_curve

from src import config, features, labeling, modeling

SEED = 42
BOOTSTRAP_REPLICATES = 2000

## Chronological split, boundary checks and metrics

In [2]:
def split_cohort(cohort):
    ordered = cohort.sort_values("first_ts")
    val_start = int(ordered.iloc[int(len(ordered) * 0.6)].first_ts)
    test_start = int(ordered.iloc[int(len(ordered) * 0.8)].first_ts)
    idx = cohort.first_ts
    train = cohort[(idx < val_start) & (cohort.cp_end <= val_start)]
    val = cohort[(idx >= val_start) & (idx < test_start) & (cohort.cp_end <= test_start)]
    test = cohort[idx >= test_start]
    assert train.cp_end.max() <= val.first_ts.min()
    assert val.cp_end.max() <= test.first_ts.min()
    assert sum(len(x) for x in (train, val, test)) == len(set().union(
        *(set(x.player_id) for x in (train, val, test))))
    return train, val, test

def verify_window_boundaries():
    day = labeling.DAY
    events = pd.DataFrame({
        "player_id": ["early", "retained", "retained", "churn", "churn", "late", "end"],
        "ts": [0, 6*day, 11*day, 8*day, 23*day, 29*day, 30*day],
    })
    cohort = labeling.build_cohort(events).set_index("player_id")
    assert set(cohort.index) == {"retained", "churn"}
    assert cohort.loc["retained", ["n_events_op","n_events_cp","churn"]].tolist() == [1,1,0]
    assert cohort.loc["churn", ["n_events_op","n_events_cp","churn"]].tolist() == [1,0,1]

def group_metrics(y, score, threshold):
    pred = score >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "n": len(y), "retained": int(tn+fp), "auc": float(roc_auc_score(y, score)),
        "average_precision_retained": float(average_precision_score(1-y, 1-score)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "retained_correct": int(tn), "churn_correct": int(tp),
        "fpr": float(fp/(fp+tn)), "fnr": float(fn/(fn+tp)),
    }

def auc_gap_interval(y, score, play_count, heavy_min):
    light = play_count == 1
    heavy = play_count >= heavy_min
    cells = [np.flatnonzero(group & (y == label)) for group in (light, heavy) for label in (0,1)]
    rng = np.random.default_rng(SEED)
    gaps = []
    for _ in range(BOOTSTRAP_REPLICATES):
        ln, lp, hn, hp = [rng.choice(cell, len(cell), replace=True) for cell in cells]
        li, hi = np.r_[ln, lp], np.r_[hn, hp]
        gaps.append(roc_auc_score(y[hi],score[hi])-roc_auc_score(y[li],score[li]))
    return np.quantile(gaps, [0.025, 0.975]).tolist()


def paired_metric_intervals(y, logistic_score, recency_score, prevalence):
    """Resample paired predictions while preserving counts for both classes."""
    by_class = [np.flatnonzero(y == label) for label in (0, 1)]
    rng = np.random.default_rng(SEED + 1)
    auc_differences = []
    brier_differences = []
    paired_error = (y - prevalence)**2 - (y - logistic_score)**2
    for _ in range(BOOTSTRAP_REPLICATES):
        idx = np.concatenate([rng.choice(rows, len(rows), replace=True) for rows in by_class])
        auc_differences.append(
            roc_auc_score(y[idx], logistic_score[idx]) - roc_auc_score(y[idx], recency_score[idx])
        )
        brier_differences.append(paired_error[idx].mean())
    return {
        "roc_auc_full_minus_recency": np.quantile(auc_differences, [0.025, 0.975]).tolist(),
        "brier_prevalence_minus_full": np.quantile(brier_differences, [0.025, 0.975]).tolist(),
    }


## Fit models, choose thresholds on validation and evaluate the final test cohort

In [3]:
def main():
    verify_window_boundaries()
    events = pd.read_parquet(config.INTERIM_DIR / "game1_events.parquet")
    assert len(events) == 153_929 and events.player_id.nunique() == 25_956
    cohort = labeling.build_cohort(events)
    all_features = features.build_features(events, cohort, config.OBSERVATION_PERIOD_DAYS)
    assert len(cohort) == 25_121 and all_features.player_id.is_unique
    assert np.array_equal(all_features.set_index("player_id").play_count.sort_index(),
                          cohort.set_index("player_id").n_events_op.sort_index())
    cohorts = split_cohort(cohort)
    player_features = all_features.set_index("player_id", drop=False)
    train, val, test = [player_features.loc[part.player_id] for part in cohorts]
    columns = features.feature_columns(features.prune_features(train))
    y_train, y_val, y_test_series = (frame.churn for frame in (train, val, test))
    y_test = y_test_series.to_numpy()
    models = modeling.build_models(y_train)
    models["random_forest"].set_params(clf__n_jobs=4)
    models["xgboost"].set_params(n_jobs=4)
    models["recency_only"] = modeling.build_models(y_train)["logistic_regression"]
    output = {
        "packages": {name: importlib.metadata.version(name) for name in
                     ("numpy", "pandas", "scikit-learn", "xgboost", "pyarrow")},
        "python": platform.python_version(),
        "bootstrap_seed": SEED,
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "events": len(events), "eligible": len(cohort), "columns": columns,
        "purged": len(cohort)-sum(map(len, cohorts)),
        "splits": [
            {"n": len(part), "retained": int((part.churn == 0).sum()),
             "first": str(pd.to_datetime(part.first_ts.min(), unit="s").date()),
             "last": str(pd.to_datetime(part.first_ts.max(), unit="s").date())}
            for part in cohorts
        ],
        "prevalence_baseline": float(y_train.mean()),
        "baseline_brier": brier_score_loss(y_test, np.full(len(y_test), y_train.mean())),
        "baseline_ap": float((1-y_test).mean()),
        "models": {},
    }
    test_scores = {}
    for name, model in models.items():
        inputs = ["recency_hours"] if name == "recency_only" else columns
        model.fit(train[inputs], y_train)
        val_raw = model.predict_proba(val[inputs])[:, 1]
        test_raw = model.predict_proba(test[inputs])[:, 1]
        calibration = LogisticRegression(C=1000, max_iter=1000).fit(val_raw[:, None], y_val)
        val_score = calibration.predict_proba(val_raw[:, None])[:, 1]
        test_score = calibration.predict_proba(test_raw[:, None])[:, 1]
        fpr, tpr, thresholds = roc_curve(y_val, val_score, drop_intermediate=False)
        threshold = float(thresholds[1:][np.argmax((tpr-fpr)[1:])])
        test_scores[name] = test_score
        metrics = group_metrics(y_test, test_score, threshold)
        metrics.update(threshold=threshold, brier_raw=brier_score_loss(y_test,test_raw),
                       brier_calibrated=brier_score_loss(y_test,test_score),
                       validation_auc=roc_auc_score(y_val,val_score))
        output["models"][name] = metrics
        if name == "logistic_regression":
            play_count = test.play_count.to_numpy()
            comparisons = {}
            for heavy_min in (3, 4):
                light = play_count == 1
                heavy = play_count >= heavy_min
                comparisons[heavy_min] = {
                    "light": group_metrics(y_test[light],test_score[light],threshold),
                    "heavy": group_metrics(y_test[heavy],test_score[heavy],threshold),
                    "gap_interval": auc_gap_interval(y_test,test_score,play_count,heavy_min),
                }
            output["engagement_sensitivity"] = comparisons
        print(name, metrics, flush=True)
    output["paired_intervals"] = paired_metric_intervals(
        y_test, test_scores["logistic_regression"], test_scores["recency_only"], y_train.mean()
    )
    (ROOT / "game1_temporal_results.json").write_text(
        json.dumps(output, indent=2) + "\n", encoding="utf-8")

In [4]:
main()
results = json.loads((ROOT / "game1_temporal_results.json").read_text(encoding="utf-8"))
print("Chronological cohorts:", results["splits"])
print("Training prevalence baseline Brier score:", results["baseline_brier"])
print("Engagement comparisons:", json.dumps(results["engagement_sensitivity"], indent=2))
print("Paired comparisons:", results["paired_intervals"])

  cohort: 5 players -> 4 after left guard -> 2 after right guard -> 2 with OP activity | churn rate 50.0%


  cohort: 25,956 players -> 25,422 after left guard -> 25,121 after right guard -> 25,121 with OP activity | churn rate 92.6%


  dropped 2 uninformative columns: std_gap_hours, activity_slope


logistic_regression {'n': 5025, 'retained': 307, 'auc': 0.7832329715152861, 'average_precision_retained': 0.3261235131192991, 'balanced_accuracy': 0.7331089058053363, 'retained_correct': 200, 'churn_correct': 3844, 'fpr': 0.3485342019543974, 'fnr': 0.18524798643493007, 'threshold': 0.9504016747326276, 'brier_raw': 0.15579548177132393, 'brier_calibrated': 0.04966654159426571, 'validation_auc': 0.7876552854747512}


random_forest {'n': 5025, 'retained': 307, 'auc': 0.7368871450802457, 'average_precision_retained': 0.2901279239203027, 'balanced_accuracy': 0.671375686434792, 'retained_correct': 166, 'churn_correct': 3784, 'fpr': 0.4592833876221498, 'fnr': 0.19796523950826622, 'threshold': 0.9349555305361479, 'brier_raw': 0.07679016227818246, 'brier_calibrated': 0.05048185608268009, 'validation_auc': 0.7311954317007618}


xgboost {'n': 5025, 'retained': 307, 'auc': 0.7428943556660818, 'average_precision_retained': 0.28022580913516837, 'balanced_accuracy': 0.6883769001661113, 'retained_correct': 159, 'churn_correct': 4052, 'fpr': 0.4820846905537459, 'fnr': 0.14116150911403136, 'threshold': 0.9284376414791211, 'brier_raw': 0.11635328084230423, 'brier_calibrated': 0.05105066311807502, 'validation_auc': 0.7281265314284835}


recency_only {'n': 5025, 'retained': 307, 'auc': 0.7811072156948301, 'average_precision_retained': 0.30152953952840245, 'balanced_accuracy': 0.7363103120214634, 'retained_correct': 206, 'churn_correct': 3782, 'fpr': 0.3289902280130293, 'fnr': 0.19838914794404408, 'threshold': 0.9694885342796923, 'brier_raw': 0.16324624944047073, 'brier_calibrated': 0.0505365842454327, 'validation_auc': 0.7820421537021289}


Chronological cohorts: [{'n': 14548, 'retained': 1228, 'first': '2015-01-18', 'last': '2015-08-30'}, {'n': 4139, 'retained': 233, 'first': '2015-09-14', 'last': '2015-12-22'}, {'n': 5025, 'retained': 307, 'first': '2016-01-06', 'last': '2016-05-09'}]
Training prevalence baseline Brier score: 0.05790560799543439
Engagement comparisons: {
  "3": {
    "light": {
      "n": 2401,
      "retained": 51,
      "auc": 0.5386733416770964,
      "average_precision_retained": 0.03143918456003585,
      "balanced_accuracy": 0.4997872340425532,
      "retained_correct": 0,
      "churn_correct": 2349,
      "fpr": 1.0,
      "fnr": 0.000425531914893617
    },
    "heavy": {
      "n": 1611,
      "retained": 215,
      "auc": 0.7679916039181716,
      "average_precision_retained": 0.42060505923898167,
      "balanced_accuracy": 0.6659009129073099,
      "retained_correct": 187,
      "churn_correct": 645,
      "fpr": 0.13023255813953488,
      "fnr": 0.5379656160458453
    },
    "gap_interval": 